# Customer Segmentation ETL

## Purpose
Segment customers by purchase behavior for business analytics and targeted marketing.

## Input → Output
* **Source 1:** `big_data.silver.orders`
* **Source 2:** `big_data.silver.order_products` 
* **Source 3:** `big_data.silver.products_enriched` 
* **Target:** `big_data.gold.dt_customer_segmentation`
* **Primary Key:** `user_id`

## Transformations
1. Load and Aggregate Customer Data - JOIN orders with order_products and products, GROUP BY user_id to calculate metrics (total_orders, total_items, estimated_lifetime_value_usd, avg_days_between_orders, reordered_items)
2. Create Customer Segments - Assign purchase_frequency_segment (Very Active, Active, Regular, Moderate, Occasional) based on avg_days_between_orders percentiles, add timestamp

## Data Quality
* **Technical:** Customer count > 200K, NOT NULL (user_id, total_orders), Lifecycle metrics (1 aggregated row)
* **Business:** All 5 segments present, Segment distribution, Lifetime values > 0

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Target table (dimension table with dt_ prefix)
target_table = "dt_customer_segmentation"

# Primary Key columns for validation
primary_key_columns = ["user_id"]

# Critical columns (NOT NULL required)
critical_columns = ["total_orders"]

# Expected segments for validation (based on actual data distribution)
expected_segments = ["Very Active", "Active", "Regular", "Moderate", "Occasional"]

# Expected metrics
expected_metrics = {
    "min_customers": 200_000
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Target: {gold_schema}.{target_table}")
print(f"  Primary Key: {', '.join(primary_key_columns)}")

### TRANSFORMATION

In [0]:
print("Step 1: Loading and aggregating...")

orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")
products = spark.table(f"{silver_schema}.products_enriched")

enriched = orders.join(order_products, "order_id").join(products.select("product_id", "price_usd"), "product_id", "left")

customer_agg = enriched.groupBy("user_id").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.count("product_id").alias("total_items"),
    F.round(F.sum("price_usd"), 2).alias("estimated_lifetime_value_usd"),
    F.round(F.avg(F.when(F.col("days_since_prior_order").isNotNull(), F.col("days_since_prior_order"))), 2).alias("avg_days_between_orders"),
    F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
)

print(f"  Customers: {customer_agg.count():,}")

In [0]:
print("Step 2: Creating customer segments and lifecycle metrics...")

# Create customer segmentation with purchase frequency segments
# Segments based on data distribution percentiles:
# - Very Active: 0-9 days (0-25th percentile)
# - Active: 10-14 days (25-50th percentile)  
# - Regular: 15-20 days (50-75th percentile)
# - Moderate: 21-25 days (75-90th percentile)
# - Occasional: 26-30 days (90-100th percentile)
customer_segmentation_gold = customer_agg \
    .withColumn(
        "purchase_frequency_segment",
        F.when(F.col("avg_days_between_orders") <= 9, "Very Active")
         .when(F.col("avg_days_between_orders") <= 14, "Active")
         .when(F.col("avg_days_between_orders") <= 20, "Regular")
         .when(F.col("avg_days_between_orders") <= 25, "Moderate")
         .otherwise("Occasional")
    ) \
    .withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Customer segmentation: {customer_segmentation_gold.count():,} customers")

# Create overall lifecycle metrics
customer_lifecycle_gold = customer_segmentation_gold.agg(
    F.countDistinct("user_id").alias("total_customers"),
    F.round(F.avg("total_orders"), 2).alias("avg_orders_per_customer"),
    F.round(F.avg("estimated_lifetime_value_usd"), 2).alias("avg_customer_lifetime_value_usd")
).withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Lifecycle metrics: 1 aggregated row")

print("\nPreview - Customer Segmentation (Sample 5):")
customer_segmentation_gold.orderBy(F.desc("estimated_lifetime_value_usd")).show(5, truncate=False)

print("\nPreview - Customer Lifecycle Metrics:")
customer_lifecycle_gold.show(1, truncate=False, vertical=True)

In [0]:
# Create final DataFrame for validation and persistence
df_result = customer_segmentation_gold

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
print_validation_header("Customer Segmentation - Technical Validations")

# Initialize validation flag
validation_technical = True

# 1. Customer count check
customer_count = df_result.count()
print(f"\nTotal customers: {customer_count:,}")
print(f"Expected: >= {expected_metrics['min_customers']:,}\n")

if customer_count >= expected_metrics["min_customers"]:
    status = "PASS"
    msg = f"Customer count ({customer_count:,}) >= {expected_metrics['min_customers']:,}"
else:
    status = "FAIL"
    msg = f"Customer count ({customer_count:,}) < {expected_metrics['min_customers']:,}"
    validation_technical = False
print_check_result("CUSTOMER COUNT", status, msg)

# 2. NOT NULL checks
print("\n2. NOT NULL Validations:")
status, failed, msg = check_not_null(df_result, primary_key_columns + critical_columns)
print_check_result("NOT NULL (user_id, total_orders)", status, msg, failed)
if status == "FAIL":
    validation_technical = False

# 3. Lifecycle metrics check
lifecycle_count = customer_lifecycle_gold.count()
if lifecycle_count == 1:
    status = "PASS"
    msg = "Lifecycle metrics: 1 aggregated row"
else:
    status = "FAIL"
    msg = f"Lifecycle metrics: {lifecycle_count} rows (expected 1)"
    validation_technical = False
print_check_result("LIFECYCLE METRICS ROW COUNT", status, msg)

total_rows = customer_count

print("\n" + "="*60)
if validation_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("Customer Segmentation - Business Validations")

# Initialize business validation flag
validation_business = True

# 1. All segments present
print("\n1. Business Rule - Segment Coverage:")
actual_segments = [r.purchase_frequency_segment for r in df_result.select("purchase_frequency_segment").distinct().collect()]
missing_segments = set(expected_segments) - set(actual_segments)

if len(missing_segments) == 0:
    status = "PASS"
    msg = f"All {len(expected_segments)} segments present: {sorted(actual_segments)}"
else:
    status = "FAIL"
    msg = f"Missing segments: {missing_segments}"
    validation_business = False
print_check_result("SEGMENT COVERAGE (5 segments)", status, msg)

# 2. Segment distribution
print("\n2. Business Rule - Segment Distribution:")
segment_dist = df_result.groupBy("purchase_frequency_segment").count().orderBy(F.desc("count")).collect()
print("\n  Distribution:")
for seg in segment_dist:
    print(f"    - {seg['purchase_frequency_segment']}: {seg['count']:,} customers ({seg['count']/total_rows*100:.1f}%)")

# 3. Lifetime value validation
print("\n3. Business Rule - Lifetime Value:")
zero_value_customers = df_result.filter(F.col("estimated_lifetime_value_usd") <= 0).count()

if zero_value_customers == 0:
    status = "PASS"
    msg = "All customers have positive lifetime value"
else:
    status = "FAIL"
    msg = f"{zero_value_customers} customers with non-positive lifetime value"
    validation_business = False
print_check_result("LIFETIME VALUE > 0", status, msg, zero_value_customers)

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, f"{gold_schema}.{target_table}")
    
    # Also persist lifecycle metrics (separate aggregated table)
    print("\nPersisting customer lifecycle metrics...")
    customer_lifecycle_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.ft_customer_lifecycle_metrics")
    
    print(f"\n✓ Lifecycle metrics table: {gold_schema}.ft_customer_lifecycle_metrics (1 row)")
    print("\nNext Step: Query for customer insights and segmentation analysis")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")